In [0]:
from typing import Dict, List, Optional, Any
import os
import re
from pathlib import Path

In [0]:
%run "../helpers/000-log-helper"

In [0]:
%run "../helpers/000-databricks-sql-executor"

In [0]:
# Get the name of the notebook
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_name = nb_path.split("/")[-1]

# Instantiate the helper class to get the Spark session and logger
helper = HELPER()
spark = helper.get_spark_session(nb_name)
logger = helper.get_logger(nb_name)

# SQL Executor
executor = DatabricksSQLExecutor(spark)

In [0]:
# parameters
dbutils.widgets.text("env", "")
dbutils.widgets.text("schema_name", "")
env = dbutils.widgets.get("env")

if env == "":
    env = "dev"
else:
    env = dbutils.widgets.get("env")

schema_name = dbutils.widgets.get("schema_name")
if not schema_name:
    raise logger.error(f"Schema Name Must Be Specified")

In [0]:
# SQL files - list all the files

sql_dir = Path("/Workspace/Repos/development/datavault-datamodelling/datascripts/schema")
sql_files = [file for file in sql_dir.iterdir() if file.is_file() and file.suffix == ".sql"]


# Catalog Schema
# env = 'dev'
catalog_name = env
# schema_name = "metadataregistry"

# execute sql files
for file in sql_files:
    logger.info(f"Starting execution for {file.name} [env={env}, schema={schema_name}]")
    try:
        executor.execute_sql_file(file, {"env": env, "catalog_name": env, "schema_name": schema_name})
        logger.info(f"Successfully executed {file.name}")
    except Exception as e:
        logger.error(f"Failed execution for {file.name}: {e}")